Cell 1: Setup & Load Gold

In [0]:
# MAGIC %md
# MAGIC # ⚡ EV Site Selection — NB05: Export & Final Report
# MAGIC **Objective:** สร้างแผนที่ Interactive, สรุปผล KPI และ Export CSV สำหรับ Streamlit Dashboard

%pip install folium

import os
import folium
from folium.plugins import MarkerCluster
import pandas as pd
from pyspark.sql.functions import col, desc

DATABASE_NAME = "ev_site_selection"
spark.sql(f"USE {DATABASE_NAME}")

df_gold  = spark.table("gold_grid_scored")
pdf_gold = df_gold.orderBy(desc("total_score")).limit(100).toPandas()
print(f"✅ Loaded {len(pdf_gold)} top locations for visualization")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
✅ Loaded 100 top locations for visualization


Cell 2: Executive Summary

In [0]:
# MAGIC %md
# MAGIC ## Cell 2: Executive Summary

total_grids = df_gold.count()
golden_gaps = df_gold.filter(col("is_golden") == True).count()
best_loc    = df_gold.orderBy("payback_years").select("grid_lat", "grid_lon", "payback_years").first()
avg_payback = df_gold.filter(col("is_viable") == True).agg({"payback_years": "avg"}).collect()[0][0]

print("━" * 58)
print("  📊 Executive Summary")
print("━" * 58)
print(f"  Total grids analyzed     : {total_grids:,}")
print(f"  Golden Gap locations     : {golden_gaps}  (viable + flood-safe)")
print(f"  Avg payback (viable set) : {avg_payback:.2f} years")
print(f"  Best payback location    : {best_loc['payback_years']:.2f} yr "
      f"({best_loc['grid_lat']}, {best_loc['grid_lon']})")
print("━" * 58)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  📊 Executive Summary
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Total grids analyzed     : 18,492
  Golden Gap locations     : 2252  (viable + flood-safe)
  Avg payback (viable set) : 3.37 years
  Best payback location    : 1.51 yr (13.7695, 100.6376)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Cell 3: Interactive Folium Map

In [0]:
# MAGIC %md
# MAGIC ## Cell 3: Interactive Folium Map
# MAGIC 🟢 Golden Gap | 🟡 High Demand + flood risk | 🔴 Others

BKK_CENTER = [13.7563, 100.5018]
m = folium.Map(location=BKK_CENTER, zoom_start=11, tiles="CartoDB positron")

for _, row in pdf_gold.iterrows():
    if row["is_golden"]:
        color = "green"
    elif row["total_score"] > 0.5 and not row["is_flood_safe"]:
        color = "orange"
    else:
        color = "red"

    popup_html = f"""
    <b>Zone:</b> {row['zone_type']}<br>
    <b>Payback:</b> {row['payback_years']} yr<br>
    <b>Score:</b> {row['total_score']:.3f}<br>
    <b>Flood Risk:</b> {'Low ✅' if row['is_flood_safe'] else 'High ❌'}
    """
    folium.Marker(
        location=[row["grid_lat"], row["grid_lon"]],
        popup=folium.Popup(popup_html, max_width=200),
        icon=folium.Icon(color=color, icon="bolt", prefix="fa")
    ).add_to(m)

display(m)
m.save("ev_site_optimization_map.html")
print("✅ Map saved → ev_site_optimization_map.html")


✅ Map saved → ev_site_optimization_map.html


Cell 4: Cluster & ROI Validation 

In [0]:
# MAGIC %md
# MAGIC ## Cell 4: Cluster & ROI Validation

print("📊 Cluster distribution and average ROI by zone:")
validation_df = df_gold \
    .groupBy("cluster", "zone_type") \
    .agg({"payback_years": "avg", "total_score": "avg", "*": "count"}) \
    .orderBy("cluster")

display(validation_df)


📊 Cluster distribution and average ROI by zone:


cluster,zone_type,avg(payback_years),avg(total_score),count(1)
0,CBD,2.9235714285714303,0.1995783173389795,84
0,URBAN,74.49332101047384,0.022439363121095103,8115
1,URBAN,97.82248394863565,-0.3603630292696379,2492
1,CBD,93.41235294117647,-0.33209241850366333,34
2,URBAN,68.67863986947492,0.03195975348400933,6742
2,CBD,35.41417560975596,0.12027367796758935,1025


Cell 5: Export CSV for Streamlit Dashboard

In [0]:
# MAGIC %md
# MAGIC ## Cell 5: Export gold_data.csv for Streamlit Dashboard

current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
volume_path     = f"/Volumes/{current_catalog}/{DATABASE_NAME}/mlflow_tmp_volume"

spark.sql(f"CREATE VOLUME IF NOT EXISTS {current_catalog}.{DATABASE_NAME}.mlflow_tmp_volume")

file_path = f"{volume_path}/gold_data.csv"

try:
    df_gold.filter(col("is_golden") == True) \
           .toPandas() \
           .to_csv(file_path, index=False)
    print(f"✅ Exported gold_data.csv → {file_path}")
    print("🚀 Ready for Streamlit / Hugging Face deployment")
except Exception as e:
    print(f"❌ Export failed: {e}")

✅ Exported gold_data.csv → /Volumes/workspace/ev_site_selection/mlflow_tmp_volume/gold_data.csv
🚀 Ready for Streamlit / Hugging Face deployment
